In [58]:
import torch
import matplotlib as plt
import torch.nn.functional as F

In [59]:
words = open("names.txt", "r").read().splitlines()

In [60]:
len(words)

32033

In [61]:
characters = sorted(list(set("".join(words))))

In [62]:
stoi = {ch: i + 1 for i, ch in enumerate(characters)}
stoi["."] = 0

itos = {i: ch for ch, i in stoi.items()}

In [63]:
freq_tables = torch.zeros(27, 27, dtype=torch.int32)

In [64]:
for word in words:
    new_word = "." + word + "."
    for ix in range(len(new_word) - 1):
        first_ch, second_ch = new_word[ix], new_word[ix + 1]
        first_ix, second_ix = stoi[first_ch], stoi[second_ch]
        freq_tables[first_ix][second_ix] += 1

In [70]:
has_zero = (freq_tables == 0).any()
print(has_zero)

tensor(True)


In [72]:
p = freq_tables.float()
p /= p.sum(1, keepdim=True)

In [66]:
sum(p[0, :])

tensor(1.)

In [67]:
g = torch.Generator().manual_seed(42)
for _ in range(10):
    out = []
    idx = 0

    while True:
        idx = torch.multinomial(p[idx], num_samples=1, replacement=True, generator=g).item()
        out.append(itos[idx])
        if idx == 0:
            break
    print("".join(out))

anugeenvi.
s.
mabian.
dan.
stan.
silaylelaremah.
li.
le.
epiachalen.
diza.


In [68]:
n = 0
loss = 0

for word in words[:3]:
    word = "." + word + "."
    for idx in range(len(word) - 1):
        n += 1
        x, y = word[idx], word[idx + 1]
        x_idx, y_idx = stoi[x], stoi[y]
        prob = p[x_idx, y_idx]
        if prob == 0:
            print("ERROR")
        loss -= torch.log(prob)

print(f"avg loss: {loss / n}")
    

avg loss: 2.424102306365967


In [54]:
xs, ys = [], []
for word in words:
    chs = ["."] + list(word) + ["."]
    for idx in range(len(chs) - 1):
        xs.append(stoi[chs[idx]])
        ys.append(stoi[chs[idx + 1]])

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

In [55]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [57]:
epoch = 50000
for k in range(epoch):
    xenc = F.one_hot(xs, num_classes=27).float()
    logit = xenc @ W
    # probs = F.softmax(logit)
    counts = logit.exp()
    probs = counts / counts.sum(1, keepdim=True) # softmax function
    loss = -probs[torch.arange(num), ys].log().mean() # negative log likelihood
    print(loss.item())
    # backward
    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

    

2.456974744796753
2.456967830657959
2.4569613933563232
2.4569549560546875
2.4569482803344727
2.456941604614258
2.456935167312622
2.4569287300109863
2.4569222927093506
2.456915855407715
2.4569091796875
2.4569029808044434
2.4568963050842285
2.456890106201172
2.456883668899536
2.4568774700164795
2.4568710327148438
2.456864833831787
2.4568588733673096
2.456852436065674
2.456846237182617
2.4568405151367188
2.456834077835083
2.4568281173706055
2.4568216800689697
2.4568159580230713
2.4568099975585938
2.456804037094116
2.4567978382110596
2.456792116165161
2.4567861557006836
2.456780195236206
2.4567744731903076
2.456768274307251
2.4567625522613525
2.456756830215454
2.4567508697509766
2.4567453861236572
2.4567394256591797
2.4567339420318604
2.456727981567383
2.4567222595214844
2.456716775894165
2.4567112922668457
2.4567055702209473
2.456700086593628
2.4566943645477295
2.45668888092041
2.456683397293091
2.4566779136657715
2.456672430038452
2.456666946411133
2.4566614627838135
2.456655979156494
2.

KeyboardInterrupt: 

In [76]:
xs = torch.tensor([4, 1, 2, 3])
xs = F.one_hot(xs, 5)

In [78]:
W = torch.tensor([
    [1, 2, 1],
    [2, 1, 1],
    [3, 2, 1],
    [1, 2, 3],
    [2, 2, 2]
])
xs @ W

tensor([[2, 2, 2],
        [2, 1, 1],
        [3, 2, 1],
        [1, 2, 3]])

In [83]:
torch.tensor([-1111111]).exp()

tensor([0.])